# [저널 확장 · 2026-10-27] 학습된 관성과 시작 상태 효과 — 공식의 사전 예측 검증 (EIIE)

**질문.** 명시적 스무딩이 없는 공개 구조(EIIE, Jiang et al. 2017 — 직전 비중을 입력으로 받음)도 거래비용이 크면
스스로 관성을 학습하는가? 그렇다면 시작 상태 효과가 닫힌 형태 공식
`효과 ≈ (x₀ − x̄) · Σ_k (1 − α)^k · log(1 + b_k)` 으로 **테스트 데이터를 보기 전에** 예측되는가?

**사전 등록(실행 전 커밋).**

| 항목 | 값 |
|---|---|
| 모델 | EIIE(원 노트북 구조·하이퍼파라미터 그대로), 학습 2025-01-01~09-30, 100,000 스텝 |
| 조건 | 거래비용 tc ∈ {0.001, 0.005, 0.02} × 시드 {42, 7} = 6개 모델. 학습·평가 모두 같은 tc |
| 관성 추정(학습 구간만) | 학습 구간에 고르게 놓은 시작점 20개에서 현금 시작·등가중 시작으로 14일 실행 → 두 경로의 노출 차이 \|x_EW − x_cash\| 평균을 k = 0~335 스텝에서 log 선형 회귀 → α_eff = 1 − exp(기울기). x̄ = 같은 시작점들에서 2주 워밍업 뒤 14일 평균 노출 |
| 예측(테스트 보기 전 확정) | 테스트(2026-01-01~07-13) 28일 창, 7일 간격. 시작 {등가중(x₀=1), 현금(x₀=0)} 각각 예측 = (x₀ − x̄) · Σ (1 − α_eff)^k · log(1 + b_k), b = 거래 가능 종목 등가중 30분 수익 |
| 관측 | 같은 창에서 정책을 그 시작 상태로 실행한 log 성장 − 2주 워밍업 후 실행한 log 성장 |
| H4 (학습된 관성) | α_eff가 tc가 클수록 작아짐: 6개 모델에서 tc와 α_eff의 스피어만 상관 ≤ −0.6 |
| H5 (공식 예측) | α_eff < 0.5인 모델들의 (창 × 시작) 쌍을 합쳐 관측 vs 예측 상관 ≥ 0.8 이고 회귀 기울기 ∈ [0.7, 1.3]. 해당 모델이 없으면 '판정 불가(학습된 관성 없음)' |

- **`런타임 > 모두 실행`만 누르면 된다.** CPU로 약 3~5시간(학습 6회).
- **끊겨도 다시 `모두 실행`만 누르면 이어서 진행한다.** 학습은 약 1만 스텝마다 Drive에 중간 저장되어 마지막 저장 지점부터 이어 학습하고(난수 상태는 이어지지 않으므로 끊긴 모델은 연속 학습과 비트 단위로 같지는 않다), 평가는 모델별로 저장되어 끝난 모델은 건너뛴다.
- 드라이브 연결 창에서 **ksbdaniel7@gmail.com** 계정을 고른다.
- 결과: `data/crypto/rt1006/learned_inertia_alpha.csv`, `learned_inertia_windows.csv`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q stable-baselines3[extra] gymnasium h5py pyarrow


In [ ]:
# 실행기 — GitHub의 원본 노트북(커밋 0e86a41 고정)을 불러와 셀을 순서대로 실행한다.
# (1) 설정 셀 직후 모델·결과 경로를 새 폴더(rt1006)로 바꿔 기존 결과를 덮어쓰지 않는다.
# (2) 모든 .learn() 호출 결과를 캐시에 저장 — 끊겨도 `모두 실행`을 다시 누르면 학습된 부분은 건너뛴다.
import json, urllib.request
from pathlib import Path
COMMIT = '0e86a41a701841d9a17398a8cebf240cd0c40218'
RAW = f'https://raw.githubusercontent.com/winbean-kwon/ToBeRich/{COMMIT}/'
TAG = 'rt1006'
_ip = get_ipython()

def fetch_cells(path):
    nb = json.loads(urllib.request.urlopen(RAW + path).read().decode('utf-8'))
    return {i: (c['source'] if isinstance(c['source'], str) else ''.join(c['source']))
            for i, c in enumerate(nb['cells']) if c['cell_type'] == 'code'}

PATCH = r"""
MODEL_DIR = DRIVE_ROOT / 'models' / TAG
CRYPTO_DIR = DATA_DIR / 'crypto' / TAG
MODEL_DIR.mkdir(parents=True, exist_ok=True); CRYPTO_DIR.mkdir(parents=True, exist_ok=True)
POOL_DIR = MODEL_DIR / 'crypto_hrl_pool'
ORIG_PPO_PATH = MODEL_DIR / 'crypto_ppo_portfolio.zip'
for _n in ('EIIE_MODEL_DIR', 'LSRE_CAAN_MODEL_DIR'):
    if _n in globals():
        globals()[_n] = MODEL_DIR / Path(globals()[_n]).name
        globals()[_n].mkdir(parents=True, exist_ok=True)
CACHE = MODEL_DIR / '_learn_cache'; CACHE.mkdir(exist_ok=True)
import stable_baselines3 as _sb3
_BASE_PPO, _BASE_DQN = _sb3.PPO, _sb3.DQN
def _cached(base):
    class _C(base):
        def learn(self, total_timesteps, *a, **k):
            _CTR[0] += 1
            p = CACHE / f'{_NB[0]}_c{_CUR[0]}_{_CTR[0]}_{base.__name__}.zip'
            if p.exists():
                self.set_parameters(str(p), exact_match=True, device=self.device)
                self.num_timesteps += total_timesteps
                print(f'[캐시] {p.name} 로드 — 학습 생략')
                return self
            out = super().learn(total_timesteps, *a, **k)
            self.save(str(p))
            return out
    _C.__name__ = _C.__qualname__ = base.__name__
    return _C
PPO, DQN = _cached(_BASE_PPO), _cached(_BASE_DQN)
print(f'[경로] 모델 → {MODEL_DIR} | 결과 CSV → {CRYPTO_DIR}')
"""

_NB, _CUR, _CTR = [''], [0], [0]

def run_cells(path, ids, patch_after, extra=None, subs=None):
    """extra: 경로 변경 직후 실행할 코드(예: β 변경). subs: {셀번호: [(원문, 대체), ...]} 문자열 치환."""
    cells = fetch_cells(path)
    _NB[0] = Path(path).stem
    for i in ids:
        _CUR[0], _CTR[0] = i, 0
        src = cells[i]
        for old, new in (subs or {}).get(i, []):
            assert old in src, f'셀 {i}에 치환 대상 없음: {old!r}'
            src = src.replace(old, new)
        print(f'\n########## {Path(path).name} 셀 {i} ##########')
        r = _ip.run_cell(src)
        if not r.success:
            raise RuntimeError(f'{path} 셀 {i}에서 오류 — 위 오류 메시지를 확인하세요')
        if i == patch_after:
            for code in (PATCH, extra):
                if code and not _ip.run_cell(code).success:
                    raise RuntimeError('경로/설정 변경 실패')
print('실행기 준비 완료')


In [ ]:
# EIIE 준비 — 원본 노트북(커밋 0e86a41) 셀을 그대로 실행(학습·평가 셀 제외)
run_cells('crypto/notebooks/crypto_eiie_baseline_colab.ipynb', [3, 5, 7, 9, 11, 13, 15], patch_after=3)
import numpy as np, pandas as pd
LI_DIR = MODEL_DIR / 'learned_inertia'; LI_DIR.mkdir(parents=True, exist_ok=True)
TCS, SEEDS = [0.001, 0.005, 0.02], [42, 7]
TRAIN_END = '2025-09-30'
li_train = [b for b in all_buckets if RL_TRAIN_START <= bucket_to_date(b) <= TRAIN_END]
li_test = [b for b in all_buckets if '2026-01-01' <= bucket_to_date(b) <= '2026-07-13']
H, WARM, WIN, STEP = 14 * 48, 14 * 48, 28 * 48, 7 * 48
print(f'학습 {len(li_train)} · 테스트 {len(li_test)} 스텝')


In [ ]:
%%time
# 학습 — 완성 모델이 있으면 건너뛰고, 중간 저장이 있으면 거기서 이어서 학습한다
import re
from stable_baselines3 import PPO as _PPO
from stable_baselines3.common.callbacks import CheckpointCallback
CKPT_FREQ = 10_240   # PPO rollout(1,024) 10회마다 저장
for tc in TCS:
    for seed in SEEDS:
        p = LI_DIR / f'eiie_tc{tc}_s{seed}.zip'
        if p.exists():
            print('있음:', p.name); continue
        ck = LI_DIR / f'ckpt_tc{tc}_s{seed}'; ck.mkdir(exist_ok=True)
        env = Monitor(EIIEPortfolioEnv(li_train, symbols, close_mat, high_mat, low_mat, pos_of, tc=tc))
        saved = sorted(ck.glob('m_*_steps.zip'), key=lambda f: int(re.search(r'_(\d+)_steps', f.name).group(1)))
        if saved:
            m = _PPO.load(str(saved[-1]), env=env, custom_objects={'policy_kwargs': policy_kwargs}, device='cpu')
            print(f'이어서 학습: {saved[-1].name} ({m.num_timesteps:,} 스텝부터)', flush=True)
        else:
            m = _PPO('MlpPolicy', env, learning_rate=3e-4, n_steps=1024, batch_size=64, n_epochs=10, gamma=0.99,
                     gae_lambda=0.95, clip_range=0.2, ent_coef=0.01, vf_coef=0.5, max_grad_norm=0.5,
                     policy_kwargs=policy_kwargs, verbose=0, device='cpu', seed=seed)
        remain = EIIE_TIMESTEPS - m.num_timesteps
        if remain > 0:
            m.learn(total_timesteps=remain, reset_num_timesteps=False,
                    callback=CheckpointCallback(save_freq=CKPT_FREQ, save_path=str(ck), name_prefix='m'))
        m.save(str(p)); print('저장:', p.name, flush=True)


In [ ]:
%%time
# 관성 추정(학습 구간) → 테스트 예측 확정 → 관측
_probe = EIIEPortfolioEnv(li_test[:2], symbols, close_mat, high_mat, low_mat, pos_of)


def basket_log(bks):
    out = []
    for b in bks:
        r = np.asarray(_probe._asset_ret(b), dtype=np.float64)[:-1]; t = np.asarray(_probe._tradable(b), dtype=bool)[:-1]
        out.append(np.log1p(r[t].mean() if t.any() else 0.0))
    return np.asarray(out)


def run(model, tc, bks, start, warm_from=None):
    # start: 'cash' / 'ew' / 'warmup'(warm_from 버킷 목록을 현금에서 먼저 실행). 반환: (log 성장, 스텝별 노출)
    run_bk = (list(warm_from) if start == 'warmup' else []) + list(bks)
    skip = len(warm_from) if start == 'warmup' else 0
    env = EIIEPortfolioEnv(run_bk, symbols, close_mat, high_mat, low_mat, pos_of, tc=tc)
    obs, _ = env.reset()
    n = env.n_assets
    if start == 'ew':
        t = np.asarray(env._tradable(run_bk[0]), dtype=np.float32)[:n]
        w = np.zeros(n + 1, dtype=np.float32); w[:n] = t / max(t.sum(), 1.0); env.weights = w; obs = env._get_obs()
    vals, expo, done = [float(env.portfolio_value)], [], False
    while not done:
        a, _ = model.predict(obs, deterministic=True)
        obs, _, done, _, info = env.step(a)
        vals.append(float(env.portfolio_value)); expo.append(1.0 - float(env.weights[-1]))
    return float(np.log(vals[-1] / vals[skip])), np.asarray(expo[skip:])


# 이어서 실행: 이미 평가가 끝난 모델은 저장된 결과를 불러오고 건너뛴다(모델 하나가 끝날 때마다 저장됨)
AP, WP = CRYPTO_DIR / 'learned_inertia_alpha.csv', CRYPTO_DIR / 'learned_inertia_windows.csv'
alpha_rows = pd.read_csv(AP).to_dict('records') if AP.exists() else []
win_rows = pd.read_csv(WP).to_dict('records') if WP.exists() else []
DONE = {(float(r['tc']), int(r['seed'])) for r in alpha_rows}
win_rows = [w for w in win_rows if (float(w['tc']), int(w['seed'])) in DONE]
print('평가 완료된 모델:', sorted(DONE))
starts_tr = np.linspace(WARM, len(li_train) - H - 1, 20).astype(int)
for tc in TCS:
    for seed in SEEDS:
        if (float(tc), int(seed)) in DONE:
            print(f'건너뜀(완료): tc={tc} seed={seed}'); continue
        model = PPO.load(str(LI_DIR / f'eiie_tc{tc}_s{seed}.zip'), custom_objects={'policy_kwargs': policy_kwargs})
        gaps, xs = [], []
        for s in starts_tr:
            bk = li_train[s:s + H]
            _, e_ew = run(model, tc, bk, 'ew'); _, e_cash = run(model, tc, bk, 'cash')
            _, e_w = run(model, tc, bk, 'warmup', warm_from=li_train[s - WARM:s])
            m = min(len(e_ew), len(e_cash)); gaps.append(np.abs(e_ew[:m] - e_cash[:m])[:336]); xs.append(e_w.mean())
        g = np.mean(np.stack([x[:336] for x in gaps]), axis=0)
        k = np.arange(len(g)); ok = g > 1e-4
        slope = np.polyfit(k[ok], np.log(g[ok]), 1)[0] if ok.sum() > 10 else 0.0
        a_eff, xbar = float(np.clip(1 - np.exp(slope), 1e-4, 1.0)), float(np.mean(xs))
        alpha_rows.append({'tc': tc, 'seed': seed, 'α_eff': a_eff, 'x̄(학습)': xbar, '초기 노출 차이': float(g[0])})
        print(f'tc={tc} seed={seed}: α_eff={a_eff:.4f}, x̄={xbar:.3f}, 초기 차이 {g[0]:.3f}', flush=True)
        # 예측은 테스트 실행 전에 모두 확정
        preds = []
        for s in range(WARM, len(li_test) - WIN + 1, STEP):
            bl = basket_log(li_test[s:s + WIN]); d = (1 - a_eff) ** np.arange(len(bl))
            for st, x0 in (('ew', 1.0), ('cash', 0.0)):
                preds.append((s, st, float((x0 - xbar) * (d * bl).sum())))
        for s, st, pred in preds:
            g_s, _ = run(model, tc, li_test[s:s + WIN], st)
            g_w, _ = run(model, tc, li_test[s:s + WIN], 'warmup', warm_from=li_test[s - WARM:s])
            win_rows.append({'tc': tc, 'seed': seed, '창 시작': str(bucket_to_ts(li_test[s])), '시작': st,
                             '예측': pred, '관측': g_s - g_w})
        pd.DataFrame(win_rows).to_csv(WP, index=False, encoding='utf-8-sig')
        pd.DataFrame(alpha_rows).to_csv(AP, index=False, encoding='utf-8-sig')   # 완료 표시는 alpha 파일 — 창 결과를 먼저 저장
print('저장 완료')


In [ ]:
# 사전 등록 판정
from scipy.stats import spearmanr
A = pd.DataFrame(alpha_rows); W = pd.DataFrame(win_rows)
print(A.round(4).to_string(index=False))
rho = spearmanr(A['tc'], A['α_eff']).correlation
print(f"\n[H4] tc vs α_eff 스피어만 {rho:+.3f} → {'지지' if rho <= -0.6 else '지지 안 됨'}")
inert = A[A['α_eff'] < 0.5][['tc', 'seed']]
Wi = W.merge(inert, on=['tc', 'seed'])
if len(inert) == 0:
    print('[H5] 판정 불가 — α_eff < 0.5인 모델 없음(학습된 관성 없음)')
else:
    r = np.corrcoef(Wi['예측'], Wi['관측'])[0, 1]; sl = np.polyfit(Wi['예측'], Wi['관측'], 1)[0]
    ok = r >= 0.8 and 0.7 <= sl <= 1.3
    print(f"[H5] 관성 모델 {len(inert)}개, 쌍 {len(Wi)}개: 상관 {r:.3f}, 기울기 {sl:.2f} → {'지지' if ok else '지지 안 됨'}")
    print(Wi.groupby(['tc', 'seed']).apply(lambda g: pd.Series({'상관': np.corrcoef(g['예측'], g['관측'])[0, 1],
          '평균|관측| 로그%p': g['관측'].abs().mean() * 100})).round(3).to_string())
